<a href="https://colab.research.google.com/github/karthikkumarp2416sse-bit/ATM-simulator/blob/main/ex_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

def calculate_entropy(data):
    class_column = data.columns[-1]
    class_counts = data[class_column].value_counts()
    total_samples = len(data)
    entropy = 0
    for count in class_counts:
        probability = count / total_samples
        entropy -= probability * np.log2(probability)
    return entropy

def calculate_information_gain(data, attribute):
    total_entropy = calculate_entropy(data)
    weighted_entropy = 0
    attribute_values = data[attribute].unique()
    for value in attribute_values:
        subset = data[data[attribute] == value]
        weighted_entropy += (len(subset) / len(data)) * calculate_entropy(subset)
    return total_entropy - weighted_entropy

In [ ]:
def build_id3_tree(data, attributes, target_attribute):
    if len(data[target_attribute].unique()) == 1:
        return data[target_attribute].iloc[0]

    if len(attributes) == 0:
        return data[target_attribute].mode()[0]

    if data.empty:
        return None

    gains = {attr: calculate_information_gain(data, attr) for attr in attributes}
    best_attribute = max(gains, key=gains.get)

    tree = {best_attribute: {}}

    for value in data[best_attribute].unique():
        subset = data[data[best_attribute] == value].drop(columns=[best_attribute])
        remaining_attributes = [attr for attr in attributes if attr != best_attribute]
        tree[best_attribute][value] = build_id3_tree(subset, remaining_attributes, target_attribute)

    return tree

In [ ]:
data = {
    'Outlook': ['Sunny', 'Sunny', 'Overcast', 'Rain', 'Rain', 'Rain', 'Overcast', 'Sunny', 'Sunny', 'Rain', 'Sunny', 'Overcast', 'Overcast', 'Rain'],
    'Temperature': ['Hot', 'Hot', 'Hot', 'Mild', 'Cool', 'Cool', 'Cool', 'Mild', 'Cool', 'Mild', 'Mild', 'Mild', 'Hot', 'Mild'],
    'Humidity': ['High', 'High', 'High', 'High', 'Normal', 'Normal', 'Normal', 'High', 'Normal', 'Normal', 'Normal', 'High', 'Normal', 'High'],
    'Wind': ['Weak', 'Strong', 'Weak', 'Weak', 'Weak', 'Strong', 'Strong', 'Weak', 'Weak', 'Weak', 'Strong', 'Strong', 'Weak', 'Strong'],
    'Play Tennis': ['No', 'No', 'Yes', 'Yes', 'Yes', 'No', 'Yes', 'No', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'No']
}
df = pd.DataFrame(data)

attributes = ['Outlook', 'Temperature', 'Humidity', 'Wind']
target_attribute = 'Play Tennis'

id3_tree = build_id3_tree(df, attributes, target_attribute)

import json
print(json.dumps(id3_tree, indent=2))

{
  "Outlook": {
    "Sunny": {
      "Humidity": {
        "High": "No",
        "Normal": "Yes"
      }
    },
    "Overcast": "Yes",
    "Rain": {
      "Wind": {
        "Weak": "Yes",
        "Strong": "No"
      }
    }
  }
}


In [ ]:
def classify(sample, tree):
    if not isinstance(tree, dict):
        return tree

    attribute = list(tree.keys())[0]
    attribute_value = sample[attribute]

    if attribute_value in tree[attribute]:
        return classify(sample, tree[attribute][attribute_value])
    else:
        return None

new_sample = {
    'Outlook': 'Sunny',
    'Temperature': 'Cool',
    'Humidity': 'High',
    'Wind': 'Strong'
}

prediction = classify(new_sample, id3_tree)

print(f"New Sample: {new_sample}")
print(f"Predicted 'Play Tennis': {prediction}")

New Sample: {'Outlook': 'Sunny', 'Temperature': 'Cool', 'Humidity': 'High', 'Wind': 'Strong'}
Predicted 'Play Tennis': No
